# 04 - Detection Inference and Crop Generation
Use trained YOLO model to detect panels and generate crops for downstream analysis.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

DETECTION_ROOT = PROJECT_ROOT / "rooftop-solar-panels-object-detection"
CLASSIFICATION_ROOT = PROJECT_ROOT / "rooftop-solar-panels-image-classification" / "Faulty_solar_panel"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DETECTION_ROOT exists:", DETECTION_ROOT.exists())
print("CLASSIFICATION_ROOT exists:", CLASSIFICATION_ROOT.exists())


PROJECT_ROOT: C:\Users\DigvijayYadav\Downloads\rooftop-solar-panel-dataset
DETECTION_ROOT exists: True
CLASSIFICATION_ROOT exists: True


In [2]:
from detection_utils import run_detection_and_generate_crops

# Auto-resolve latest trained YOLO best weights
candidate_weights = sorted(
    (PROJECT_ROOT / "artifacts/models").glob("det_yolov8n_cpu*/weights/best.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
weights = candidate_weights[0] if candidate_weights else None

# Prefer test split; fallback to val/train/original images if needed
candidate_input_dirs = [
    PROJECT_ROOT / "data/processed/yolo_single_class/images/test",
    PROJECT_ROOT / "data/processed/yolo_single_class/images/val",
    PROJECT_ROOT / "data/processed/yolo_single_class/images/train",
    DETECTION_ROOT / "images",
]
input_dir = next((d for d in candidate_input_dirs if d.exists() and any(d.iterdir())), None)
out_dir = PROJECT_ROOT / "data/processed/detected_crops"

print("Resolved weights:", weights)
print("Resolved input_dir:", input_dir)

if weights is None:
    raise FileNotFoundError("No YOLO best.pt found under artifacts/models/det_yolov8n_cpu*/weights/")
if input_dir is None:
    raise FileNotFoundError("No non-empty input image directory found for detection inference")

crops_df = run_detection_and_generate_crops(weights, input_dir, out_dir, conf=0.25)
print("Generated crops:", len(crops_df))
display(crops_df.head())

out_csv = PROJECT_ROOT / "data/processed/detected_crops_index.csv"
crops_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)



Resolved weights: C:\Users\DigvijayYadav\Downloads\rooftop-solar-panel-dataset\artifacts\models\det_yolov8n_cpu3\weights\best.pt
Resolved input_dir: C:\Users\DigvijayYadav\Downloads\rooftop-solar-panel-dataset\data\processed\yolo_single_class\images\test
Generated crops: 162


,crop_id,parent_image_id,crop_path,bbox_confidence,x1,y1,x2,y2
0,015032f6-00000127_rgb_resized_0,015032f6-00000127_rgb_resized,C:\Users\DigvijayYadav\Downloads\rooftop-solar...,0.862945,0,195,338,385
1,015032f6-00000127_rgb_resized_1,015032f6-00000127_rgb_resized,C:\Users\DigvijayYadav\Downloads\rooftop-solar...,0.311577,5,193,472,389
2,02e12757-Portable-solar-panel-by-camper_0,02e12757-Portable-solar-panel-by-camper,C:\Users\DigvijayYadav\Downloads\rooftop-solar...,0.908891,332,280,1256,900
3,077a9956-00000177_rgb_resized_0,077a9956-00000177_rgb_resized,C:\Users\DigvijayYadav\Downloads\rooftop-solar...,0.591165,23,52,412,364
4,08eb68f3-00000162_rgb_resized_0,08eb68f3-00000162_rgb_resized,C:\Users\DigvijayYadav\Downloads\rooftop-solar...,0.839648,2,190,480,444


Saved: C:\Users\DigvijayYadav\Downloads\rooftop-solar-panel-dataset\data\processed\detected_crops_index.csv
